# SAM ViT-B — DIMER promptable image segmentation tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/sam-vit-segmentation-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/sam-vit-segmentation-pipeline/blob/main/tutorials/sam_vit_segmentation_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-facebook%2Fsam--vit--base-ffcc4d?style=flat)](https://huggingface.co/facebook/sam-vit-base)
[![Upstream](https://img.shields.io/badge/Upstream-facebookresearch%2Fsegment--anything-181717?style=flat&logo=github&logoColor=white)](https://github.com/facebookresearch/segment-anything)
[![arXiv](https://img.shields.io/badge/arXiv-2304.02643-b31b1b.svg)](https://arxiv.org/abs/2304.02643)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** promptable image segmentation (point and/or box prompts → masks for one object) using the pinned `facebook/sam-vit-base` weights (SAM v1, ViT-B)

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API rather than reimplementing model inference. At inference the image's longest edge is resized to 1024 px and padded to 1024×1024, encoded once by a ViT-B image encoder; the prompt encoder embeds your clicks (label 1 = foreground, 0 = background) and/or one xyxy box, and the mask decoder returns up to three candidate masks at 256×256, which the pipeline up-samples to the input resolution (stripping the padding) and binarises at logit 0, together with the model's own predicted IoU for each candidate. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights and processor, and this repository adds packaging, snapshot verification, image and prompt validation with named ceilings, a single-object-per-call contract, a fixed output contract and the `mask_iou` helper. This is the original 2023 SAM; the sibling `sam2-segmentation-pipeline` packages SAM 2.1 with the same contract, and the two are not compared here. The default sample is a synthetic scene drawn in code; the IoU reported for it is sanity evidence against a shape you drew, not a benchmark claim.

**Learning objectives:** bootstrap the repository in a fresh runtime, resolve the immutable upstream model revision through the package's staging and verification path, draw a synthetic scene with a known object, validate the image and prompts against the pipeline's ceilings, segment one object from a click through the public API, read the candidate masks and their model-predicted IoU scores correctly (including why a score can exceed 1.0), check the best mask against the drawn shape with `mask_iou` as sanity evidence, exercise an optional BYOD path, and export the mask plus machine-readable provenance.

**This notebook does not demonstrate:** automatic "segment everything" mask generation (the upstream grid-prompt pipeline is not wrapped), text prompts (see the sibling Grounding DINO pipeline for boxes from text), mask-input prompts, several objects in one call, semantic class labels, video, mIoU evaluation against labelled masks, or any training. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 3.77 s to load (after a 0.21 s manifest verification) and 2.93 s for the click prompt on the 320×240 synthetic scene in the Windows venv (Intel Core Ultra 9 275HX); the encoder cost is fixed by the 1024×1024 working size, so image resolution only changes the size of the returned masks (a 4096×4096 box prompt took 3.11 s in the same smoke). The pinned `torch==2.14.0` install and the 375 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python, NumPy and PIL; what a binary mask is; what intersection-over-union measures.
- **Data:** the default sample is a deterministic 320×240 scene drawn in code (grey background, one dark rectangle, one red disc) with a foreground click inside the rectangle, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image file decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, sides between 16 and 4096 px, plus a click position inside it set through the form parameters. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** GitHub (repository clone), PyPI (pinned wheels) and the Hugging Face Hub (the package's `stage_missing_files` fetches the git-ignored `model.safetensors` once at the immutable revision, because the Git repository does not vendor the weights). No credentials are required.

## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `transformers`, `safetensors`, `huggingface-hub`, `numpy`, `pillow`) are pinned exactly in `pyproject.toml`. If installation replaces any package that this runtime has already imported (hosted runtimes commonly pre-import a different NumPy or Pillow), the cell fails with a restart instruction rather than continuing with mixed versions: restart the runtime and rerun from the top. Look for a dictionary reporting the repository revision, Python, `torch` and `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/sam-vit-segmentation-pipeline.git'
REPO_NAME = 'sam-vit-segmentation-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, torch, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Draw the synthetic scene or optional BYOD

The default sample is **synthetic** and carries its own reference: a deterministic 320×240 RGB scene is drawn in code — grey background, a dark filled rectangle at `[40, 60, 140, 180]` and a red filled disc at `[200, 80, 280, 160]` — and the prompt is one **foreground click at (90, 120)**, inside the rectangle. A boolean reference mask of the drawn rectangle is kept for the IoU sanity check later. This is the same scene and click the repository's card-pass smoke used; it is not a labelled dataset, so nothing here is an mIoU measurement. The image digest and the prompt are printed. BYOD is optional and disabled by default; when enabled, upload one image and set `POINT_X`/`POINT_Y` to a pixel inside the object you want (no reference mask exists for it, so no IoU is computed).

Prompts describe **one object per call**: up to `MAX_PROMPTS` (16) clicks with 0/1 labels and/or one xyxy box; several objects need several calls. `MULTIMASK` (default `True`) asks for three candidate masks — useful when a single click is ambiguous (part, object, or object plus surroundings) — while `False` returns one. Before anything expensive runs, this cell surfaces the pipeline's operational ceilings — `MIN_IMAGE_SIDE` (16 px), `MAX_IMAGE_SIDE` (4096 px), `MAX_PROMPTS` (16), `NUM_MULTIMASK_OUTPUTS` (3) — and checks the image sides and that the click lies inside the image with clear messages; the pipeline's own `validate_prompts` repeats the check before inference. Inside the pipeline the image is converted to RGB, resized and padded by the processor; masks are mapped back to input pixels with the padding stripped, and nothing else is dropped or altered. Look for a dictionary naming the sample kind, image size and digest, the click, the multimask setting, and the reference mask area.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

from sam_vit_segmentation_pipeline import MAX_IMAGE_SIDE, MAX_PROMPTS, MIN_IMAGE_SIDE, NUM_MULTIMASK_OUTPUTS

USE_BYOD = False  # @param {type:"boolean"}
POINT_X = 90  # @param {type:"integer"}
POINT_Y = 120  # @param {type:"integer"}
MULTIMASK = True  # @param {type:"boolean"}

print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PROMPTS': MAX_PROMPTS, 'NUM_MULTIMASK_OUTPUTS': NUM_MULTIMASK_OUTPUTS}})
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    reference_mask = None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic scene: no randomness, so no seed is needed and the digest is stable.
    image = Image.new('RGB', (320, 240), (128, 128, 128))
    draw = ImageDraw.Draw(image)
    rectangle_box = [40, 60, 140, 180]
    draw.rectangle(rectangle_box, fill=(30, 30, 30))
    draw.ellipse([200, 80, 280, 160], fill=(220, 30, 30))
    reference = Image.new('1', image.size, 0)
    ImageDraw.Draw(reference).rectangle(rectangle_box, fill=1)
    reference_mask = np.asarray(reference, dtype=np.bool_)
    image_name = 'synthetic_scene_320x240.png'
    sample_kind = 'synthetic'

width, height = image.size
if min(width, height) < MIN_IMAGE_SIDE or max(width, height) > MAX_IMAGE_SIDE:
    raise ValueError(f'{image_name}: image {image.size} must have both sides within {MIN_IMAGE_SIDE}..{MAX_IMAGE_SIDE} px; resize it and rerun this cell.')
if not (0 <= POINT_X < width and 0 <= POINT_Y < height):
    raise ValueError(f'click ({POINT_X}, {POINT_Y}) lies outside the {width}x{height} image; set POINT_X/POINT_Y inside the object and rerun this cell.')
points, point_labels = [[POINT_X, POINT_Y]], [1]
image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'size': image.size, 'rgb_sha256': image_sha256, 'points': points, 'point_labels': point_labels, 'multimask': MULTIMASK, 'reference_mask_area_px': None if reference_mask is None else int(reference_mask.sum())})

## 3. Stage, verify and resolve the pinned model

Model acquisition goes through the package, not the notebook. The public API pins the exact upstream revision (`MODEL_ID`/`MODEL_REVISION` are imported from the package, never typed here). The Git repository carries `weights/sam-vit-base/dimer-base-manifest.json` (model id, revision, and the byte size and SHA-256 of each of the 4 snapshot files), `config.json`, `preprocessor_config.json` and the upstream `README.md`, but git-ignores the 375 MB `model.safetensors`, so in a fresh clone `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches exactly the manifest entries that are absent, at the pinned revision, into the snapshot directory — it prints the list it fetched (`['model.safetensors']` on a fresh clone, `[]` on a warm runtime) and refuses a manifest whose identity differs from the package pins. `verify_snapshot(WEIGHTS_DIR)` then re-hashes every manifest entry (size and SHA-256) and raises on the first mismatch; its returned summary dict (path, model id, revision, file count, total bytes) is printed. Only then does `from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files with `local_files_only=True` and `trust_remote_code=False` — there is no fallback to a different download. The upstream repository's `pytorch_model.bin` and TensorFlow weights are not in the manifest and are never staged or loaded. The card-pass smoke wrote nothing to stderr during the load, so no loader notice is expected; a warning or an error here is a real signal. The effective model identity and the device chosen (`cuda:0` when available, else `cpu`) are printed before inference.

In [ ]:
from sam_vit_segmentation_pipeline import MODEL_ID, MODEL_KEY, MODEL_REVISION, SAMViTSegmentationPipeline, mask_iou, stage_missing_files, verify_snapshot
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
print(snapshot)
pipe = SAMViTSegmentationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device})

## 4. Segment one object and read the scores correctly

`segment` returns a dict with `masks` — a boolean array of shape `(K, H, W)` at input resolution, `K = 3` with `multimask=True` else `1` — `iou_scores` (one per mask), the cleaned `points`, `point_labels` and `box`, `multimask`, `width`, `height` and the model identity. Each `iou_scores` entry is the **model's own prediction** of how well that candidate overlaps the intended object: a learned, **uncalibrated** estimate, not a measured IoU and not a probability. It is an unclipped regression output, so **a score can exceed 1.0** — the card-pass smoke returned `1.012` for the best candidate on this very scene — and a value above 1 means nothing more than "ranked highest". The conventional decision rule — used below — is to keep the candidate with the highest predicted IoU; the pipeline ships no threshold, does not choose for you, and the caller owns any acceptance rule for their deployment. Masks are binarised at logit 0 (the processor default). Inference is deterministic on a fixed device and dtype (no sampling, `torch.inference_mode`); CUDA kernel selection can move scores in the third or fourth decimal place and, on ambiguous clicks, change which candidate ranks first.

No segmentation metric is reported: mean IoU needs labelled masks, and this repository ships none. The only helper is `mask_iou(a, b)` (intersection-over-union of two boolean masks), the primitive a caller would use to compute mIoU on their own labelled data. On the synthetic path it compares the best candidate with the reference mask of the drawn rectangle, and the mask area is compared with the rectangle's area, as a **sanity check** that the prompt, forward pass and up-sampling round-trip; on BYOD no reference exists and none is computed. As recorded in the model card, the repository's CPU smoke on this same scene and click returned `iou_scores` of `[0.955, 1.012, 0.979]`, mask areas of `[17129, 12216, 11887]` px against the rectangle's 12,221 px, argmax candidate 1, and `mask_iou` 1.000 at three decimals, in 2.93 s; that is one observation, not a calibration point. Look for a best-mask area close to the reference area and a `mask_iou` near 1 on the synthetic scene; a materially different result on your runtime is a signal to check the install, not a measurement.

In [ ]:
result = pipe.segment(image, points=points, point_labels=point_labels, multimask=MULTIMASK)
masks = result['masks']
best = int(np.argmax(result['iou_scores']))
best_mask = masks[best]
print({'masks_shape': masks.shape, 'iou_scores_model_predicted': [round(v, 4) for v in result['iou_scores']], 'scores_above_one': [i for i, v in enumerate(result['iou_scores']) if v > 1.0], 'mask_areas_px': [int(m.sum()) for m in masks], 'best_candidate': best, 'device': pipe.device})
metrics = {}
if reference_mask is not None:
    metrics['sanity_vs_drawn_rectangle'] = {'best_candidate': best, 'best_mask_area_px': int(best_mask.sum()), 'reference_area_px': int(reference_mask.sum()), 'mask_iou_best_vs_reference': mask_iou(best_mask, reference_mask)}
    print({'sample_metrics': metrics, 'note': 'IoU against a rectangle you drew yourself on a synthetic scene: sanity evidence, not a segmentation metric'})
else:
    print('No reference mask exists for a BYOD image, so mask_iou is not computed; inspect the exported mask and overlay instead.')

## 5. Export outputs and provenance

The best candidate mask is written as a 1-bit PNG (`outputs/sam_vit_segmentation_mask.png`) — the actual artifact a downstream consumer wants — and an overlay PNG paints it over the input for visual inspection (a supplement to, not a replacement for, the machine-readable files). JSON preserves every candidate's model-predicted IoU and area, the chosen candidate, the prompt, the sanity block when computed, the sample identity and digest, the verified snapshot summary, the repository revision, the model identifier, the immutable model revision, and the runtime identity (Python, `torch`, `transformers`, device). No credentials are recorded.

In [ ]:
import json
os.makedirs('outputs', exist_ok=True)
Image.fromarray(best_mask).save('outputs/sam_vit_segmentation_mask.png')
overlay = np.asarray(image.convert('RGB')).copy()
overlay[best_mask] = (0.5 * overlay[best_mask] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)
Image.fromarray(overlay, mode='RGB').save('outputs/sam_vit_segmentation_overlay.png')
payload = {
    'prediction': {key: value for key, value in result.items() if key != 'masks'},
    'candidates': [{'index': i, 'iou_score_model_predicted': float(result['iou_scores'][i]), 'area_px': int(masks[i].sum())} for i in range(masks.shape[0])],
    'best_candidate': best,
    'mask_file': 'outputs/sam_vit_segmentation_mask.png',
    'mask_sha256': hashlib.sha256(np.packbits(best_mask).tobytes()).hexdigest(),
    'metrics': metrics,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'reference_mask_area_px': None if reference_mask is None else int(reference_mask.sum())},
    'snapshot': {**snapshot, 'fetched_this_run': fetched},
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/sam_vit_segmentation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The masks are the model's answer to *your* prompt: a click that is ambiguous returns candidates at several granularities, and the `iou_scores` used to rank them are the model's own uncalibrated estimates — unclipped, so occasionally above 1.0 — not measured overlaps. On the synthetic scene the IoU against the rectangle you drew proves only that the input contract, prompt validation, forward pass and up-sampling work on a trivially separable shape; it says nothing about photographs, thin structures, transparent or occluded objects, or clicks near a boundary, and a BYOD result is a single-image observation. One object per call, prompted mode only, no text prompts, no class labels. The pipeline provides no automatic mask generation, mask-input prompts, mIoU evaluation, or training capability.

Successful execution proves that the recorded repository revision can acquire the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/sam-vit-base/` and rerun Section 3. A `ValueError` naming the image sides or the click in Section 2: resize the BYOD image into 16–4096 px per side, or move `POINT_X`/`POINT_Y` inside it, and rerun from Section 2.

**Next experiments:** set `MULTIMASK = False` and compare the single mask with the best candidate; prompt the disc with a box instead of a click (`pipe.segment(image, box=[200, 80, 281, 161], multimask=False)`) and compare its area with the disc's (the smoke run returned 5,132 px against the drawn 5,145 px, `mask_iou` 0.997, `iou_scores` `[1.002]` — another score above 1.0); add a background click (label 0) inside the rectangle after a foreground click on the disc to see the mask exclude it; enable `USE_BYOD` with a photograph and hand-draw one reference mask to compute `mask_iou` yourself — the first step towards a real mIoU. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance: `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/facebook/sam-vit-base
- Upstream code: https://github.com/facebookresearch/segment-anything
- Segment Anything (Kirillov et al., ICCV 2023): https://arxiv.org/abs/2304.02643